In [1]:
import numpy as np
from scipy.stats import qmc   # qmc = quasi-Monte Carlo, has Latin Hypercube

# our 3 parameters and their physical ranges
param_names = ["d", "n1", "n2"]
lower = np.array([20.0, 1.0, 1.4])   # mins
upper = np.array([450.0, 2.5, 1.6])  # maxs

print("parameters:", param_names)
print("lower bounds:", lower)
print("upper bounds:", upper)

parameters: ['d', 'n1', 'n2']
lower bounds: [20.   1.   1.4]
upper bounds: [450.    2.5   1.6]


In [2]:
n_samples = 300   # how many points to scatter in the box

# Latin Hypercube: spread points evenly across the 3D space
sampler = qmc.LatinHypercube(d=3, seed=0)   # d=3 means 3 dimensions (our 3 params)
unit_samples = sampler.random(n=n_samples)   # points in a plain 0-to-1 cube

# stretch the 0-1 cube to our real ranges
samples = qmc.scale(unit_samples, lower, upper)

print("samples shape:", samples.shape)
print("first 3 samples:\n", samples[:3])

samples shape: (300, 3)
first 3 samples:
 [[143.78702158   1.73365107   1.52397268]
 [150.40964372   2.20593365   1.42205816]
 [403.26382205   2.41635252   1.55830425]]


In [3]:
import jax.numpy as jnp

def tmm_single_layer(n0, n1, n2, d, lam):
    # phase thickness of the layer
    delta = 2 * jnp.pi * n1 * d / lam
    # Fresnel coefficients at each interface
    r01 = (n0 - n1) / (n0 + n1)   # air -> layer
    r12 = (n1 - n2) / (n1 + n2)   # layer -> glass
    # Fabry-Perot total reflection coefficient
    r = (r01 + r12 * jnp.exp(2j * delta)) / (1 + r01 * r12 * jnp.exp(2j * delta))
    R = jnp.abs(r)**2
    return R

# quick check: bare glass (d=0) should give R ≈ 0.04
print("d=0 check:", float(tmm_single_layer(1.0, 1.5, 1.5, 0.0, 500.0)))

d=0 check: 0.04000000283122063


In [4]:
# wavelength grid for every spectrum (same 300 points as before)
wavelengths = jnp.linspace(400, 700, 300)

# for one parameter set, compute the full spectrum across wavelengths
def spectrum_for_params(d, n1, n2):
    R_at = lambda lam: tmm_single_layer(1.0, n1, n2, d, lam)  # n0=air=1.0 fixed
    return jnp.vectorize(R_at)(wavelengths)

# run TMM on all 300 sampled recipes
Y = np.array([spectrum_for_params(d, n1, n2) for d, n1, n2 in samples])

print("Y shape:", Y.shape)
print("one spectrum, first 5 wavelengths:", Y[0][:5])

Y shape: (300, 300)
one spectrum, first 5 wavelengths: [0.07542104 0.07479478 0.07417125 0.07355081 0.07293355]
